In [1]:
# Import required libraries
import os
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_2019_UGA.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,2019_UGA.ID_208,0,ENSG00000000003,NaN,0.077926,Unknown
1,2019_UGA.ID_208,0,ENSG00000000005,NaN,0.000000,Unknown
2,2019_UGA.ID_208,0,ENSG00000000419,NaN,5.342548,Unknown
3,2019_UGA.ID_208,0,ENSG00000000457,NaN,9.799336,Unknown
4,2019_UGA.ID_208,0,ENSG00000000460,NaN,2.118530,Unknown


In [3]:
# Drop raw_count (entirely NaN) and material (entirely 'Unknown') before pivoting
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2019_UGA.ID_208,0,ENSG00000000003,0.077926
1,2019_UGA.ID_208,0,ENSG00000000005,0.000000
2,2019_UGA.ID_208,0,ENSG00000000419,5.342548
3,2019_UGA.ID_208,0,ENSG00000000457,9.799336
4,2019_UGA.ID_208,0,ENSG00000000460,2.118530


In [4]:
df['timepoint'].unique()

array([ 0,  3,  7, 28])

In [5]:
# Filter to only timepoints 0, 7, 28. Timepoint 3 not in the challenge set
df = df[df['timepoint'].isin([0, 7, 28])]

# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "d{timepoint}_{gene}" format
df_pivot.columns = [f'd{int(tp)}_{gene}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)
df_pivot.head(1)

(275, 120386)


,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,...,d28_ENSG00000284575,d28_ENSG00000284583,d28_ENSG00000284584,d28_ENSG00000284585,d28_ENSG00000284586,d28_ENSG00000284587,d28_ENSG00000284594,d28_ENSG00000284595,d28_ENSG00000284596,d28_ENSG00000284600
0,2019_UGA.ID_001,0.149866,0.0,8.066946,8.457258,3.973795,95.386306,0.034801,13.583601,7.965179,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# # Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/transcriptomics_2019_UGA_cleaned.csv', index=False)